In [ ]:
def DFAReduction(initialState,finalStates,states,transitions,tsym):
    # checking for any non reachable states bfs
    reachableStates = set([initialState])
    queue = [initialState]

    while queue:
        currentState = queue.pop(0)
        for sym in tsym:
            if (currentState, sym) in transitions:
                nextState = transitions[(currentState,sym)]
                if nextState not in reachableStates:
                    reachableStates.add(nextState)
                    queue.append(nextState)

    states = states.intersection(reachableStates)
    finalStates = finalStates.intersection(reachableStates)
    transitions = {k:v for k,v in transitions.items() if k[0] in reachableStates and v in reachableStates}


    # so here we have removed all the unreachable states

    # now splitting into two groups: final and non final states
    p1 = set(finalStates)
    p2 = set(states) - p1

    temp1 = []
    if p1: 
        temp1.append(frozenset(p1))
    if p2:
        temp1.append(frozenset(p2))

    while True:
        temp2 = []
        for i in temp1:
            if len(i) == 1:
                temp2.append(i)
                continue

            sg = {}   # sub group for storing 
            for state in i:
                temp3 = []  # this will store the transitions for each state ie to which group it goes
                for sym in sorted(tsym):
                    nextState = transitions.get((state, sym))
                    nextGroupidx = None
                    if nextState:
                        for j,group in enumerate(temp1):
                            if nextState in group:
                                nextGroupidx = j
                                break
                    temp3.append((sym,nextGroupidx))
                temp3 = tuple(temp3)
                if temp3 not in sg:
                    sg[temp3] = set()
                sg[temp3].add(state)

            for g in sg.values():
                temp2.append(frozenset(g))

        if set(temp2) == set(temp1):
            break
        temp1 = temp2

    # now we have the final groups in temp1

    groupMap = {}
    for i in temp1:
        for j in i:
            groupMap[j] = i

    reducedInitialState = groupMap[initialState]
    reducedFinalStates = {groupMap[i] for i in finalStates}

    reducedTransitions = {}
    for (state,sym),nextState in transitions.items():
        reducedTransitions[(groupMap[state],sym)] = groupMap[nextState]

    return reducedInitialState,reducedTransitions,reducedFinalStates


def printDFA(Startstate,Transistions,FinalStates):
    # Starting state 
    print(f"Starting State: {set(Startstate)}")
    print("\nDFA Transitions:")
    tempfinal = set()
    for (state, symbol), nextState in Transistions.items():
        print(f"{state} --{symbol}--> {nextState}")
        if state in FinalStates:
            tempfinal.add(state)
    print("\nDFA Final States:")
    for state in tempfinal:
        print(state)
    

In [ ]:
initial_state = input("Enter the initial state: ")

temp = int(input("Enter the number of final states: "))
final_state = set()
for i in range(temp):
    final_state.add(input(f"{i+1} - Enter the final state: "))

transitions = {}
tsym = set()
states = set() 
states.add(initial_state)

temp = int(input("Enter the number of transitions: "))
for i in range(temp):
    st = input(f"{i+1} - Enter source state: ")
    st2 = input(f"{i+1} - Enter destination state: ")
    symbol = input(f"{i+1} - Enter the transition symbol: ")
    
    tsym.add(symbol)
    states.add(st)   
    states.add(st2)
    
    transitions[(st, symbol)] = st2 

reduced_start, reduced_trans, reduced_finals = DFAReduction(
    initial_state, final_state, states, transitions, tsym
)

printDFA(reduced_start, reduced_trans, reduced_finals)
